# Giorno 2 — Vehicle/People Counting & Trajectory Analysis

**Obiettivi della giornata:**
1. Conteggio veicoli con `supervision` (roboflow) su AIC21 — LineZone e PolygonZone
2. People counting su ShanghaiTech: ground truth annotations, density map, confronto con YOLO
3. Estrazione di traiettorie con `norfair` e analisi tramite heatmap

---
**Dataset:**
- **AIC21 CityFlow** — `cam_*.mp4` — telecamere di traffico urbano (AIC Track 1)
- **ShanghaiTech** — immagini statiche di folla + annotazioni ground truth (`.mat`)

**Librerie:** `supervision` (Roboflow), `norfair`, `scipy` (lettura `.mat`) + YOLOv8

> Anche oggi: solo inference su modelli pre-addestrati, nessun retraining.

## 0. Setup

In [ ]:
import os, sys

if 'google.colab' in sys.modules:
    if not os.path.exists('/content/Corso-Computer-Vision'):
        !git clone https://github.com/SalvoSamba01/Corso-Computer-Vision
    %cd /content/Corso-Computer-Vision
    !pip install -r requirements.txt
else:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])



**Import librerie Giorno 2.** Alle librerie standard si aggiungono `scipy.io` (lettura file `.mat` di ShanghaiTech) e `scipy.ndimage.gaussian_filter` (generazione density map). Da `cv_utils`: `crea_heatmap` e `sovrapponi_heatmap`.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import scipy.io as sio
from scipy.ndimage import gaussian_filter
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from huggingface_hub import hf_hub_download
import onnxruntime as ort

import sys, os
if os.path.exists('/content/Corso-Computer-Vision'):
    sys.path.append('/content/Corso-Computer-Vision/utils')
else:
    sys.path.append('../utils')
from IPython.display import Video
from cv_utils import (mostra_frame, mostra_confronto, mostra_griglia,
                       stampa_info_video, estrai_frame, crea_heatmap, sovrapponi_heatmap,
)

In [ ]:
# ── Configurazione percorsi ──────────────────────────────────────────────────
import os
from pathlib import Path

if os.path.exists('/content/Corso-Computer-Vision'):
    DATA_DIR = Path('/content/Corso-Computer-Vision/data/day_2')
    BASE_DIR  = Path('/content/Corso-Computer-Vision')
else:
    DATA_DIR = Path('../data/day_2')
    BASE_DIR  = Path('..')

AICITY_DIR   = DATA_DIR / 'AICity'
SHANGHAI_DIR = DATA_DIR / 'ShangaiTech'
IMAGES_DIR   = SHANGHAI_DIR / 'images'
GT_DIR       = SHANGHAI_DIR / 'ground-truth'

VIDEOS_DIR = BASE_DIR / 'results' / 'videos'
VIDEOS_DIR.mkdir(parents=True, exist_ok=True)

# Video AIC21 CityFlow
video_files = sorted(AICITY_DIR.glob('*.mp4'))
# Immagini ShanghaiTech
shanghai_imgs = sorted(IMAGES_DIR.glob('IMG_*.jpg'))

print(f'Video AIC21 trovati: {len(video_files)}')
for v in video_files:
    print(f'  {v.name}')
print(f'\nImmagini ShanghaiTech: {len(shanghai_imgs)}')
for f in shanghai_imgs:
    print(f'  {f.name}')
print(f'\nOutput video → {VIDEOS_DIR}')

**Caricamento YOLO.** Stesso `YOLOv8n` del Giorno 1 — i pesi sono già in cache locale (~6 MB), nessun download aggiuntivo. Usiamo lo stesso modello per garantire coerenza nei confronti.

In [ ]:
# YOLOv8n già scaricato nel Giorno 1 — pesi in cache locale (~6 MB), nessun download.
# Stesso modello del giorno precedente per garantire coerenza nei confronti.
from ultralytics import YOLO

model_yolo = YOLO('yolov8n.pt')  # pesi già scaricati ieri

---
## 1. Esplorazione del dataset AIC21

**AIC21 CityFlow** è il dataset del Track 1 dell'AI City Challenge 2021, progettato esplicitamente per testare la robustezza dei sistemi di analisi del traffico urbano.

La caratteristica più importante per i nostri scopi: **stessa telecamera, stessa scena, condizioni atmosferiche diverse**. Questo setup quasi-sperimentale è prezioso perché qualsiasi differenza nelle performance del modello è attribuibile esclusivamente alle condizioni meteo — non alla scena, all'angolazione o ai tipi di veicoli.

- **Condizioni disponibili:** normale, pioggia (`_rain`), alba (`_dawn`), neve (`_snow`)
- Le variazioni di luce (alba, notte) abbassano il contrasto e rendono i bordi dei veicoli meno definiti
- La pioggia introduce riflessi sul manto stradale che il modello può confondere con oggetti

Prima di fare analisi, esploriamo il contenuto visivo dei video per capire con cosa abbiamo a che fare.

### Metadati dei video di AIC21: risoluzione, FPS, durata per ogni condizione meteo.

In [ ]:
print('=' * 60)
for v in video_files:
    print(f'\n📹 {v.name}')
    stampa_info_video(str(v))
print('=' * 60)

### Anteprima visiva dei video AIC21

Viene mostrato il **primo frame di ciascun video** del dataset **AIC21**.

Questo permette di confrontare le differenze visive tra condizioni diverse della stessa scena, come:

- Normale  
- Pioggia  
- Alba  
- Neve

In [ ]:
frames_preview = [estrai_frame(str(v), n=1)[0] for v in video_files]
mostra_griglia(frames_preview, [v.stem for v in video_files], colonne=2)

Selezioniamo cam_12 come baseline: condizioni meteo normali, nessuna perturbazione.

In [ ]:

VIDEO_AIC21 = str(AICITY_DIR / 'cam_12.mp4')
stampa_info_video(VIDEO_AIC21)

---
## 2. Vehicle Counting con Supervision

**`supervision`** (Roboflow) è una libreria Python che semplifica il post-processing degli output YOLO: annotazione visiva, tracking integrato, e strumenti di conteggio pronti all'uso.

**LineZone — come funziona**
`LineZone` definisce una linea virtuale nel frame e conta ogni veicolo che la attraversa. La logica usa il **centroide** di ogni bounding box tracciato:
1. Ad ogni frame, il centroide di ogni track viene confrontato con la sua posizione al frame precedente
2. Se il centroide è passato da un lato all'altro della linea, si registra un attraversamento
3. La direzione dell'attraversamento (rispetto alla normale della linea) determina se contare "su" o "giu"

Grazie all'integrazione con ByteTrack, lo stesso veicolo non viene mai contato due volte anche se il suo centroide oscilla leggermente intorno alla linea.

**Flusso di lavoro:**
```
Frame video → YOLO detection → ByteTrack → LineZone.trigger() → conteggio per direzione
```

In [ ]:
import supervision as sv

### Definizione della linea di conteggio

In questa sezione definiamo la **linea di conteggio** su un frame del video **AIC21**.  

- La linea viene posizionata orizzontalmente, a metà altezza del frame, e serve come **riferimento per il conteggio degli oggetti** che la attraversano.

In [ ]:
# ── 2.1 Definizione della linea di conteggio ──────────────────────────────────
frame_cf = estrai_frame(VIDEO_AIC21, n=1)[0]
H, W = frame_cf.shape[:2]
print(f'Risoluzione: {W}x{H}')

# Disegno linea
y_centro = H // 2
LINEA_START = sv.Point(W//2, y_centro-200)
LINEA_END   = sv.Point(W,y_centro-200)

frame_con_linea = frame_cf.copy()
cv2.line(frame_con_linea,
         (LINEA_START.x, LINEA_START.y),
         (LINEA_END.x, LINEA_END.y),
         (0, 255, 255), 3)
cv2.putText(frame_con_linea, 'Linea di conteggio',
            (W//2+350, y_centro-150), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)
mostra_frame(frame_con_linea, 'Posizione della linea di conteggio')

**Configurazione supervision.** Nella pipeline di visualizzazione di **SuperVision**, si devono definire diversi componenti, ognuno dei quali ha un ruolo specifico:  
- `BoxAnnotator` — disegna il bounding box colorato attorno a ogni detection
- `LabelAnnotator` — sovrappone l'etichetta testuale (ID del track) in cima al box
- `TraceAnnotator` — disegna la scia del percorso degli ultimi N frame, rendendo visibile la direzione di movimento
- `LineZoneAnnotator` — disegna la linea di conteggio sul frame
- `ByteTrack` — il tracker che assegna ID persistenti alle detection frame per frame

`LineZone.trigger()` viene chiamato ad ogni frame e restituisce due maschere booleane — una per "su", una per "giu" — che indicano quali track hanno attraversato la linea in quel preciso frame.

In [ ]:
#Setup supervision
box_annotator   = sv.BoundingBoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_scale=0.5, text_thickness=1)
trace_annotator = sv.TraceAnnotator(thickness=2, trace_length=30)
line_zone       = sv.LineZone(start=LINEA_START, end=LINEA_END)
line_zone_ann   = sv.LineZoneAnnotator(thickness=3, text_scale=1.2, display_in_count=False, display_out_count=False)
byte_tracker    = sv.ByteTrack()

### ── 2.3 Conteggio veicoli su sequenza video ──

In [ ]:
N_FRAME_CF = 200
OUTPUT_VIDEO_CF = str(VIDEOS_DIR / 'day2_vehicle_counting.mp4')

cap = cv2.VideoCapture(VIDEO_AIC21)
fps = cap.get(cv2.CAP_PROP_FPS)
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

writer = cv2.VideoWriter(OUTPUT_VIDEO_CF, cv2.VideoWriter_fourcc(*'H264'), fps, (w, h))

conteggio_su, conteggio_giu = 0, 0

for i in tqdm(range(N_FRAME_CF), desc='Vehicle counting'):
    ret, frame = cap.read()
    if not ret:
        break

    result     = model_yolo(frame, verbose=False, classes=[2])[0]
    detections = sv.Detections.from_ultralytics(result)
    detections = byte_tracker.update_with_detections(detections)

    crossed_left, crossed_right = line_zone.trigger(detections)
    conteggio_su  += crossed_left.sum()
    conteggio_giu += crossed_right.sum()

    labels = [f'ID:{tid}' for tid in detections.tracker_id] if detections.tracker_id is not None else []
    frame = box_annotator.annotate(frame, detections)
    frame = label_annotator.annotate(frame, detections, labels)
    frame = trace_annotator.annotate(frame, detections)
    frame = line_zone_ann.annotate(frame, line_zone)

    cv2.putText(frame,
                f'Su:{conteggio_su}  Giu:{conteggio_giu}  Tot:{conteggio_su + conteggio_giu}',
                (10, 45),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.4,
                (0, 0, 255),
                3)

    writer.write(frame)

cap.release()
writer.release()
print(f'Risultati: su={conteggio_su}  giu={conteggio_giu}  totale={conteggio_su + conteggio_giu}')
print(f'Video salvato: {OUTPUT_VIDEO_CF}')
Video(OUTPUT_VIDEO_CF, embed=True)

**Risultato del counting.** Il video mostra la linea orizzontale gialla con i contatori aggiornati in tempo reale (Su / Giu / Tot). Alcuni aspetti da osservare: i veicoli nella corsia più vicina alla telecamera attraversano la linea in meno frame — appaiono più veloci non perché lo siano davvero, ma perché la prospettiva li ingrandisce. Notate anche se qualche veicolo viene contato due volte: sarebbe un segnale di ID switch nel tracker.

**Limitazione del counting con linea singola:** vengono contati tutti i veicoli che attraversano la linea, indipendentemente dalla corsia. Per un conteggio per corsia separato servirebbero più linee parallele, oppure una `PolygonZone` dedicata a ciascuna corsia.

### 2.4 PolygonZone — conteggio per area

Mentre `LineZone` conta i **transiti** (quanti oggetti hanno attraversato un confine), `PolygonZone` risponde a una domanda diversa: **quanti oggetti sono presenti *in questo momento* all'interno di una certa area?**

Sono due metriche complementari: la LineZone misura il flusso, la PolygonZone misura l'occupazione istantanea.

**Casi d'uso tipici:**
- Monitoraggio dell'occupazione di un parcheggio o di una corsia specifica
- Rilevamento di ingorghi: se il numero di veicoli nella zona supera una soglia per N frame consecutivi → alert
- Esclusione di aree non rilevanti (marciapiedi, incroci adiacenti) dall'analisi

La logica è più semplice di `LineZone`: `polygon_zone.trigger()` restituisce una maschera booleana sulle detection del frame corrente, indicando quali rientrano nel poligono. Non richiede tracking — funziona anche sulle detection grezze, senza ID persistenti.

In [ ]:
frame_ref = estrai_frame(VIDEO_AIC21, n=1)[0]
H_r, W_r = frame_ref.shape[:2]

# Zona centrale del frame — modificare le coordinate in base alla scena e alla posizione
ZONA_POLYGON = np.array([
    [W_r // 4 - 180,     H_r // 4 + 100 ],
    [3 * W_r // 4 + 240, H_r // 4 + 50],
    [3 * W_r // 4 + 400, 3 * H_r // 4 - 20],
    [W_r // 4 - 450,     3 * H_r // 4],
])

polygon_zone = sv.PolygonZone(polygon=ZONA_POLYGON)
polygon_ann  = sv.PolygonZoneAnnotator(zone=polygon_zone, color=sv.Color.GREEN, thickness=3)

frame_zona = polygon_ann.annotate(frame_ref.copy())
mostra_frame(frame_zona, 'Zona di interesse (PolygonZone)')

### Conteggio veicoli in una PolygonZone

Adesso contiamo i veicoli presenti all’interno di una **PolygonZone**.  

- `polygon_zone.trigger()` restituisce una **maschera booleana** che indica quali detection si trovano all’interno della zona in quel preciso frame, permettendo di analizzare il traffico o la densità di oggetti nell’area definita.

In [ ]:
N_FRAME_ZONA          = 100
OUTPUT_VIDEO_ZONA     = str(VIDEOS_DIR / 'day2_polygon_counting.mp4')

cap   = cv2.VideoCapture(VIDEO_AIC21)
fps_z = cap.get(cv2.CAP_PROP_FPS)
w_z   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h_z   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer = cv2.VideoWriter(OUTPUT_VIDEO_ZONA, cv2.VideoWriter_fourcc(*'H264'), fps_z, (w_z, h_z))

conteggio_zona = []

for _ in tqdm(range(N_FRAME_ZONA), desc='Counting in zona'):
    ret, frame = cap.read()
    if not ret:
        break
    result = model_yolo(frame, verbose=False, classes=[2])[0]
    dets   = sv.Detections.from_ultralytics(result)
    mask   = polygon_zone.trigger(detections=dets)
    n_in   = int(mask.sum())
    conteggio_zona.append(n_in)

    # Annotazione: zona verde + box delle detection dentro la zona + contatore
    frame = polygon_ann.annotate(frame)
    frame = box_annotator.annotate(frame, dets[mask])
    cv2.putText(frame, f'Nella zona: {n_in}', (20, 50),
            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
    writer.write(frame)

cap.release()
writer.release()

# Grafico interattivo
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(
    y=conteggio_zona, mode='lines',
    fill='tozeroy', line=dict(color='green', width=1.5),
    fillcolor='rgba(0,128,0,0.25)', name='Veicoli in zona'
))
fig.update_layout(
    title='Occupazione della zona nel tempo',
    xaxis_title='Frame', yaxis_title='Veicoli nella zona',
    height=350, margin=dict(l=50, r=20, t=50, b=40)
)
fig.show()
print(f'Media veicoli in zona: {np.mean(conteggio_zona):.1f}')
Video(OUTPUT_VIDEO_ZONA, embed=True)

---
## 3. People Counting su ShanghaiTech

### 3.1 Il dataset e le annotazioni `.mat`

**ShanghaiTech** è un benchmark classico per il **crowd counting** (conteggio di folla).
A differenza dei dataset di tracking, qui le annotazioni non sono bounding box bensì **punti**:
ogni persona è annotata con il centro della sua testa `(x, y)`.

I file `.mat` (formato MATLAB) contengono:
- `image_info.location` — array `(N, 2)` con le coordinate `(x, y)` delle teste
- `image_info.number` — conteggio totale delle persone

**Perché usare punti invece di bounding box?**
In scene molto affollate, i bounding box si sovrappongono e diventano difficili da annotare.
I punti testa sono più veloci da annotare e sufficienti per il conteggio.

In [ ]:
def carica_mat(img_nome: str) -> dict:
    """
    Carica l'annotazione ground truth per un'immagine ShanghaiTech.

    Args:
        img_nome: nome file immagine es. 'IMG_3.jpg'
    Returns:
        dict con 'n_persone' (int) e 'coordinate' (array Nx2)
    """
    num = Path(img_nome).stem.replace('IMG_', '')  # '3' da 'IMG_3'
    mat_path = GT_DIR / f'GT_IMG_{num}.mat'
    mat = sio.loadmat(str(mat_path))
    info = mat['image_info'][0, 0]
    coords = info['location'][0, 0]       # shape (N, 2): colonne = [x, y]
    n = int(info['number'][0, 0].flat[0])
    return {'n_persone': n, 'coordinate': coords}


# Test su un'immagine
esempio = carica_mat('IMG_3.jpg')
print(f'Persone annotate: {esempio["n_persone"]}')
print(f'Coordinate shape: {esempio["coordinate"].shape}')
print(f'Prime 3 annotazioni (x, y): {esempio["coordinate"][:3]}')

# ── Panoramica dei dati ──

In [ ]:
import pandas as pd

righe = []
for img_path in shanghai_imgs:
    gt = carica_mat(img_path.name)
    righe.append({'immagine': img_path.name, 'n_persone_GT': gt['n_persone']})

df_gt = pd.DataFrame(righe).sort_values('n_persone_GT')

import plotly.graph_objects as go
fig = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>Immagine</b>', '<b>Persone GT</b>'],
        fill_color='#2c3e50', font=dict(color='white', size=13),
        align='center', height=36
    ),
    cells=dict(
        values=[df_gt['immagine'].tolist(), df_gt['n_persone_GT'].tolist()],
        align='center', height=30, font=dict(size=12)
    )
)])
fig.update_layout(
    title=(f'ShanghaiTech — {len(df_gt)} immagini  '
           f'| Media: {df_gt["n_persone_GT"].mean():.0f}  '
           f'| Min: {df_gt["n_persone_GT"].min()}  '
           f'| Max: {df_gt["n_persone_GT"].max()}'),
    margin=dict(l=10, r=10, t=50, b=10), height=300
)
fig.show()

### 3.3 Visualizzazione delle annotazioni ground truth

Visualizziamo i punti testa annotati sull'immagine originale. La **ground truth** di ShanghaiTech è stata prodotta con annotazione manuale: ogni punto corrisponde a una persona identificata da un annotatore umano che ha guardato l'immagine e segnato il centro della testa.

È utile osservare visivamente la differenza tra scene a bassa e alta densità: nelle scene più affollate, i punti si sovrappongono così fittamente che diventa difficile distinguerli anche a occhio — e questo è esattamente il motivo per cui i bounding box non funzionano in questi contesti.

In [ ]:
def visualizza_gt(img_path: str, raggio: int = 5, colore=(0, 255, 0)) -> np.ndarray:
    """
    Disegna i punti testa GT sull'immagine.
    """
    img = cv2.imread(str(img_path))
    gt  = carica_mat(Path(img_path).name)
    for (x, y) in gt['coordinate']:
        cv2.circle(img, (int(x), int(y)), raggio, colore, -1)
    return img


# Visualizzazione sulle prime 4 immagini del subset
frames_gt = []
titoli_gt = []
for img_path in shanghai_imgs[:4]:
    gt = carica_mat(img_path.name)
    ann = visualizza_gt(img_path)
    frames_gt.append(ann)
    titoli_gt.append(f'{img_path.stem}\n{gt["n_persone"]} persone (GT)')

mostra_griglia(frames_gt, titoli_gt, colonne=2)

### 3.4 Density Map — dalla ground truth alla "mappa di calore delle presenze"

La **density map** è il concetto centrale del crowd counting moderno.

**Come si costruisce:**
Per ogni punto testa annotato si "piazza" una gaussiana 2D centrata su quel punto — una piccola campana di valori che decresce fino a zero allontanandosi dal centro. La density map finale è la somma di tutte queste gaussiane:
- Zone con molte persone vicine → gaussiane sovrapposte → valori alti
- Zone vuote → valori prossimi a zero
- **Proprietà fondamentale**: la somma di tutti i valori della mappa è uguale al numero esatto di persone

**Perché la gaussiana?** Modella l'incertezza spaziale dell'annotazione: il centro della testa è noto solo con una certa precisione, e la gaussiana distribuisce il "peso" della persona nell'intorno del punto annotato.

**Perché non usare YOLO sulla folla densa?**
YOLO cerca bounding box separati. In folla molto densa, le sagome si sovrappongono completamente e NMS elimina detection legittime scambiandole per duplicati. I modelli di crowd counting allo stato dell'arte (CSRNet, MCNN, BayesCrowd) sono addestrati a **predire la density map direttamente dall'immagine**, evitando completamente il problema dei box sovrapposti.

Noi la calcoliamo dalla ground truth per capire come dovrebbe apparire la predizione ideale.

In [ ]:
def crea_density_map(img_shape: tuple, coordinate: np.ndarray,
                      sigma: float = 15.0) -> np.ndarray:
    """
    Genera la density map da annotazioni puntuali.

    Args:
        img_shape: (H, W) dell'immagine
        coordinate: array (N, 2) con colonne (x, y)
        sigma: deviazione standard della gaussiana (px)
    Returns:
        density map normalizzata, shape (H, W)
    """
    H, W = img_shape[:2]
    dm = np.zeros((H, W), dtype=np.float32)

    for (x, y) in coordinate:
        xi, yi = int(np.clip(x, 0, W-1)), int(np.clip(y, 0, H-1))
        dm[yi, xi] += 1.0

    dm = gaussian_filter(dm, sigma=sigma)
    return dm

**Visualizzazione density map.** Confronto su tre colonne: immagine originale · punti GT annotati · density map sovrapposta. La mappa appare come una nebbia di calore: più intensa dove le persone sono concentrate, quasi assente nelle zone vuote. Verificate che la somma della mappa riportata nel titolo sia vicina al numero di persone annotate — questa è la proprietà che i modelli di crowd counting devono preservare nelle proprie predizioni.

In [ ]:
# ── Density map su un'immagine con alta densità ────────────────────────────────
# Scegliamo l'immagine con più persone nel subset
img_piu_densa = max(shanghai_imgs,
                     key=lambda p: carica_mat(p.name)['n_persone'])
gt_densa = carica_mat(img_piu_densa.name)
img_bgr  = cv2.imread(str(img_piu_densa))
H_sh, W_sh = img_bgr.shape[:2]

print(f'Immagine: {img_piu_densa.name}')
print(f'Persone GT: {gt_densa["n_persone"]}')

dm = crea_density_map((H_sh, W_sh), gt_densa['coordinate'], sigma=15)

# Visualizzazione affiancata
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1) Immagine originale
axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title('Immagine originale', fontsize=12)
axes[0].axis('off')

# 2) Annotazioni GT (punti testa)
img_punti = img_bgr.copy()
for (x, y) in gt_densa['coordinate']:
    cv2.circle(img_punti, (int(x), int(y)), 4, (0, 255, 0), -1)
axes[1].imshow(cv2.cvtColor(img_punti, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Punti GT ({gt_densa["n_persone"]} persone)', fontsize=12)
axes[1].axis('off')

# 3) Density map
axes[2].imshow(dm, cmap='jet', interpolation='bilinear')
axes[2].set_title(f'Density map (σ=15)\nIntegrale ≈ {dm.sum():.1f}', fontsize=12)
axes[2].axis('off')
plt.colorbar(axes[2].images[0], ax=axes[2], fraction=0.046, pad=0.04)

plt.suptitle(img_piu_densa.name, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Somma density map: {dm.sum():.1f}  (atteso: {gt_densa["n_persone"]})')

### 3.5 YOLO sulla folla densa — perché non funziona

Applichiamo YOLO alle immagini di ShanghaiTech e confrontiamo il conteggio con la ground truth.

Il problema è strutturale: YOLO cerca **bounding box separati**. In folla densa le sagome si sovrappongono così tanto che NMS (Non-Maximum Suppression) scarta detection legittime, confondendole con duplicati dello stesso oggetto. Il risultato è una sottostima sistematica che peggiora linearmente con la densità — esattamente il contrario di quello che servirebbe in un sistema di crowd monitoring.

Quantifichiamo prima il fallimento, poi introduciamo il modello corretto.

In [ ]:
# ── Conteggio YOLO su ShanghaiTech — baseline di confronto ───────────────────
SOGLIA_YOLO = 0.25  # soglia bassa per massimizzare le detection

risultati_yolo = []
for img_path in tqdm(shanghai_imgs, desc='YOLO su ShanghaiTech'):
    img = cv2.imread(str(img_path))
    gt  = carica_mat(img_path.name)
    ris = model_yolo(img, verbose=False, classes=[0], conf=SOGLIA_YOLO)
    n_yolo = len(ris[0].boxes)
    risultati_yolo.append({'immagine': img_path.name,
                            'GT':       gt['n_persone'],
                            'YOLO':     n_yolo})

df_yolo = pd.DataFrame(risultati_yolo).sort_values('GT')
mae_y   = (df_yolo['YOLO'] - df_yolo['GT']).abs().mean()
pct_y   = ((df_yolo['YOLO'] - df_yolo['GT']).abs() / df_yolo['GT'] * 100).mean()
print(f'MAE YOLO: {mae_y:.1f} persone  —  errore medio: {pct_y:.1f}%')

### 3.6 CSRNet — il modello specializzato per la folla

I risultati YOLO confermano il problema. La soluzione è usare un modello che non cerca bounding box, ma **predice direttamente la density map** dell'immagine — lo stesso tipo di rappresentazione che abbiamo costruito manualmente nella sezione 3.4.

**CSRNet** (*Dilated Convolutional Neural Networks for Understanding Highly Congested Scenes*, CVPR 2018) è il modello di riferimento per il crowd counting. L'architettura ha due parti:
- **Frontend**: i primi layer di VGG-16 estraggono le feature visive (bordi, texture, sagome)
- **Backend**: layer di *convoluzione dilatata* ampliano il campo recettivo senza ridurre la risoluzione — fondamentale per catturare pattern di folla a scale diverse

**Cos'è la convoluzione dilatata?** Una conv 3×3 normale vede i 9 pixel adiacenti. Con `dilation=2` lo stesso filtro vede un'area 5×5 "saltando" i pixel intermedi; con `dilation=4` vede 9×9. Campo recettivo grande, numero di parametri invariato — perfetto per ragionare su aree di folla piuttosto che su singoli pixel.

**Output**: density map a risoluzione 1/8 dell'input. Il conteggio = somma di tutti i valori della mappa.

Usiamo i pesi pre-addestrati su **ShanghaiTech Part A** (subset ad alta densità), disponibili su HuggingFace. Per il caricamento usiamo il formato ONNX — non richiede di definire l'architettura nel notebook.

In [ ]:
CSRNET_PATH = hf_hub_download('muasifk/CSRNet', 'model1_A.onnx')
sess_csrnet  = ort.InferenceSession(CSRNET_PATH, providers=['CPUExecutionProvider'])

inp_meta = sess_csrnet.get_inputs()[0]
out_meta = sess_csrnet.get_outputs()[0]
print(f'Input:  name={inp_meta.name}  shape={inp_meta.shape}  dtype={inp_meta.type}')
print(f'Output: name={out_meta.name}  shape={out_meta.shape}')

### Preprocessing e inferenza

Il modello **CSRNet** utilizza come frontend **VGG-16**, quindi è necessaria la **normalizzazione ImageNet**.  

- L’input deve essere nel formato **NCHW** (batch, canali, altezza, larghezza) per garantire la compatibilità con il modello durante l’inferenza.

In [ ]:
def preprocess_csrnet(img_bgr: np.ndarray, target_size=None) -> np.ndarray:
    """Preprocessa un'immagine BGR → tensor NCHW float32 normalizzato ImageNet."""
    if target_size:
        img_bgr = cv2.resize(img_bgr, target_size)
    img  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    return ((img - mean) / std).transpose(2, 0, 1)[np.newaxis]  # (1, 3, H, W)


def predici_csrnet(img_bgr: np.ndarray):
    """
    Inferenza CSRNet ONNX su un'immagine BGR.
    Gestisce sia modelli con dimensione fissa che con dimensione dinamica.
    Returns: (density_map, conteggio_predetto)
    """
    h_in = inp_meta.shape[2] if isinstance(inp_meta.shape[2], int) else None
    w_in = inp_meta.shape[3] if isinstance(inp_meta.shape[3], int) else None
    target = (w_in, h_in) if (h_in and w_in) else None

    x   = preprocess_csrnet(img_bgr, target_size=target)
    out = sess_csrnet.run(None, {inp_meta.name: x})[0]
    dm  = out[0, 0]  # density map (H/8, W/8)
    return dm, float(dm.sum())


# ── Test rapido sull'immagine più densa ────────────────────────────────────────
dm_test, count_test = predici_csrnet(cv2.imread(str(img_piu_densa)))
gt_test = carica_mat(img_piu_densa.name)
print(f'Immagine: {img_piu_densa.name}')
print(f'GT:      {gt_test["n_persone"]} persone')
print(f'CSRNet:  {count_test:.0f} persone')

Applichiamo adesso CSRNet sui dati, e confrontiamo le sue performance con quelle di YOLO e con la ground thruth

In [ ]:
risultati_csr = []
for img_path in tqdm(shanghai_imgs, desc='CSRNet su ShanghaiTech'):
    img = cv2.imread(str(img_path))
    gt  = carica_mat(img_path.name)
    _, count_csr = predici_csrnet(img)
    risultati_csr.append({'immagine': img_path.name,
                           'GT':       gt['n_persone'],
                           'CSRNet':   round(count_csr)})

df_csr = pd.DataFrame(risultati_csr)

# Dataframe unificato con entrambi i modelli
df = (df_csr
      .merge(df_yolo[['immagine', 'YOLO']], on='immagine')
      .sort_values('GT')
      .reset_index(drop=True))

df['err_yolo']   = (df['YOLO']   - df['GT']).abs()
df['err_csrnet'] = (df['CSRNet'] - df['GT']).abs()

mae_yolo   = df['err_yolo'].mean()
mae_csrnet = df['err_csrnet'].mean()

# Colore per le colonne errore: verde se CSRNet batte YOLO, arancione altrimenti
col_err_csr  = ['#c8f7c5' if r < y else '#fde3a7'
                 for r, y in zip(df['err_csrnet'], df['err_yolo'])]
col_err_yolo = ['#fde3a7' if r < y else '#c8f7c5'
                 for r, y in zip(df['err_csrnet'], df['err_yolo'])]

import plotly.graph_objects as go

fig = go.Figure(data=[go.Table(
    columnwidth=[180, 60, 60, 70, 80, 90],
    header=dict(
        values=['<b>Immagine</b>', '<b>GT</b>', '<b>YOLO</b>', '<b>CSRNet</b>',
                '<b>Err YOLO</b>', '<b>Err CSRNet</b>'],
        fill_color='#2c3e50', font=dict(color='white', size=13),
        align='center', height=36
    ),
    cells=dict(
        values=[
            df['immagine'].tolist(),
            df['GT'].tolist(),
            df['YOLO'].tolist(),
            df['CSRNet'].tolist(),
            df['err_yolo'].tolist(),
            df['err_csrnet'].tolist(),
        ],
        fill_color=['white', 'white', 'white', 'white', col_err_yolo, col_err_csr],
        align=['left', 'center', 'center', 'center', 'center', 'center'],
        height=28, font=dict(size=12)
    )
)])
fig.update_layout(
    title=(f'Confronto GT / YOLO / CSRNet — ShanghaiTech  '
           f'| MAE YOLO: {mae_yolo:.1f}  |  MAE CSRNet: {mae_csrnet:.1f} persone'),
    margin=dict(l=10, r=10, t=50, b=10),
    height=max(300, 36 + 28 * len(df) + 60)
)
fig.show()

### Grafico di confronto

In [ ]:
import plotly.graph_objects as go

lim = max(df['GT'].max(), df['CSRNet'].max(), df['YOLO'].max()) * 1.1

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df['GT'], y=df['CSRNet'], mode='markers+text',
    text=[Path(n).stem for n in df['immagine']],
    textposition='top right', textfont=dict(size=8),
    marker=dict(color='steelblue', size=9),
    name=f'CSRNet  (MAE={df["err_csrnet"].mean():.1f})',
))
fig.add_trace(go.Scatter(
    x=df['GT'], y=df['YOLO'], mode='markers',
    marker=dict(color='tomato', size=8, symbol='diamond'),
    name=f'YOLO  (MAE={df["err_yolo"].mean():.1f})',
))
fig.add_trace(go.Scatter(
    x=[0, lim], y=[0, lim], mode='lines',
    line=dict(dash='dash', color='gray', width=1),
    showlegend=False,
))
fig.update_layout(
    title='Conteggio predetto vs Ground Truth — ShanghaiTech',
    xaxis_title='Persone reali (GT)',
    yaxis_title='Persone predette',
    height=450, margin=dict(l=50, r=20, t=60, b=50),
    legend=dict(x=0.05, y=0.95),
)
fig.show()

### Confronto visivo

In [ ]:
img_bgr  = cv2.imread(str(img_piu_densa))
gt_info  = carica_mat(img_piu_densa.name)
H_img, W_img = img_bgr.shape[:2]

# YOLO boxes
ris_yolo = model_yolo(img_bgr, verbose=False, classes=[0], conf=SOGLIA_YOLO)
img_yolo = img_bgr.copy()
for box in ris_yolo[0].boxes:
    x1, y1, x2, y2 = map(int, box.xyxy[0])
    cv2.rectangle(img_yolo, (x1, y1), (x2, y2), (0, 100, 255), 2)

# GT density map
dm_gt = crea_density_map((H_img, W_img), gt_info['coordinate'], sigma=15)

# CSRNet density map (upsample alla risoluzione originale per il confronto visivo)
dm_csr, count_csr = predici_csrnet(img_bgr)
dm_csr_up = cv2.resize(dm_csr, (W_img, H_img))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(cv2.cvtColor(img_yolo, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'YOLO: {len(ris_yolo[0].boxes)} detection'
                  f'\n(GT reale: {gt_info["n_persone"]} persone)')

axes[1].imshow(dm_gt, cmap='hot')
axes[1].set_title(f'GT Density Map\n(integrale = {dm_gt.sum():.0f})')

axes[2].imshow(dm_csr_up, cmap='hot')
axes[2].set_title(f'CSRNet Density Map\n(conteggio predetto = {count_csr:.0f})')

for ax in axes:
    ax.axis('off')
plt.suptitle(f'{img_piu_densa.name} — scena con massima densita',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Osservazioni

**YOLO vs CSRNet:**
- Su scene poco affollate (< 50 persone) i due approcci danno risultati comparabili — YOLO funziona quando i bounding box non si sovrappongono
- All'aumentare della densità, l'errore YOLO cresce in modo quasi lineare; l'errore CSRNet rimane contenuto
- Confrontando le due density map: la struttura spaziale predetta da CSRNet è sorprendentemente simile alla GT — le zone dense e le zone vuote sono nelle posizioni giuste

**Limiti di CSRNet:**
- L'output è solo un numero e una mappa — nessuna informazione su dove si trova ogni singola persona
- Addestrato su una distribuzione specifica (ShanghaiTech Part A): su scene molto diverse (strade vuote, interni) le performance possono degradare
- Su scene a bassa densità YOLO rimane preferibile se servono le posizioni individuali

### 3.7 Confronto visivo: bassa · media · alta densità

Selezioniamo tre immagini del subset con densità crescente (GT minimo · medio · massimo) e confrontiamo visivamente i due approcci:

- **Ground Truth**: punti testa annotati manualmente (verde)
- **YOLO**: centroide di ogni bounding box rilevato (arancione)
- **CSRNet**: picchi locali della density map predetta (azzurro) — posizioni approssimate, il conteggio ufficiale è la somma della mappa

In [ ]:
# ── Confronto visivo: 3 livelli di densità ───────────────────────────────────
from scipy.ndimage import maximum_filter

def dm_a_punti(dm: np.ndarray, img_shape: tuple, neighborhood: int = 12) -> list:
    """
    Approssima la localizzazione delle persone dai picchi locali della density map.
    Nota: il numero di picchi NON coincide con il conteggio ufficiale (dm.sum()).
    """
    H, W = img_shape[:2]
    dm_up = cv2.resize(dm, (W, H))
    if dm_up.max() == 0:
        return []
    val_positivi = dm_up[dm_up > 0]
    if len(val_positivi) == 0:
        return []
    soglia = np.percentile(val_positivi, 85)
    peaks  = (maximum_filter(dm_up, size=neighborhood) == dm_up) & (dm_up >= soglia)
    ys, xs = np.where(peaks)
    return list(zip(xs.tolist(), ys.tolist()))


n = len(df)
esempi       = [df.iloc[0], df.iloc[n // 2], df.iloc[-1]]
label_righe  = ['BASSA densita', 'MEDIA densita', 'ALTA densita']
col_titles   = ['Ground Truth (punti testa)', 'YOLO (centroidi box)', 'CSRNet (picchi density map)']

fig, axes = plt.subplots(3, 3, figsize=(18, 16))

for row_idx, (row_df, den_label) in enumerate(zip(esempi, label_righe)):
    nome_img = row_df['immagine']
    gt       = carica_mat(nome_img)
    img_bgr  = cv2.imread(str(IMAGES_DIR / nome_img))
    img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # ── Col 1: GT punti testa ────────────────────────────────────────────────
    img_gt = img_rgb.copy()
    for (x, y) in gt['coordinate']:
        cv2.circle(img_gt, (int(x), int(y)), 3, (0, 220, 0), -1)
    axes[row_idx][0].imshow(img_gt)
    axes[row_idx][0].set_title(f'{den_label}\nGT: {gt["n_persone"]} persone')
    axes[row_idx][0].axis('off')

    # ── Col 2: YOLO — centroide di ogni bounding box ─────────────────────────
    ris      = model_yolo(img_bgr, verbose=False, classes=[0], conf=SOGLIA_YOLO)
    img_yolo = img_rgb.copy()
    n_yolo   = len(ris[0].boxes)
    for box in ris[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
        cv2.circle(img_yolo, (cx, cy), 5, (255, 100, 0), -1)
    err_y = abs(gt['n_persone'] - n_yolo) / max(gt['n_persone'], 1) * 100
    axes[row_idx][1].imshow(img_yolo)
    axes[row_idx][1].set_title(f'YOLO: {n_yolo} punti  (err: {err_y:.0f}%)')
    axes[row_idx][1].axis('off')

    # ── Col 3: CSRNet — picchi locali della density map ──────────────────────
    dm_csr, count_csr = predici_csrnet(img_bgr)
    punti_csr = dm_a_punti(dm_csr, img_bgr.shape)
    img_csr   = img_rgb.copy()
    for (x, y) in punti_csr:
        cv2.circle(img_csr, (x, y), 5, (0, 180, 255), -1)
    err_c = abs(gt['n_persone'] - count_csr) / max(gt['n_persone'], 1) * 100
    axes[row_idx][2].imshow(img_csr)
    axes[row_idx][2].set_title(f'CSRNet: count={count_csr:.0f}  (err: {err_c:.0f}%)\n'
                                f'picchi visualizzati: {len(punti_csr)}')
    axes[row_idx][2].axis('off')

# Intestazioni colonne
for col_idx, title in enumerate(col_titles):
    cur = axes[0][col_idx].get_title()
    axes[0][col_idx].set_title(f'{title}\n{cur}')

plt.suptitle('YOLO vs CSRNet — difficolta crescente', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Trajectory Extraction con Norfair

**Norfair** è un tracker leggero che espone direttamente la storia completa di ogni traccia — a differenza di ByteTrack, pensato principalmente per l'associazione veloce, Norfair è progettato per l'analisi del movimento nel tempo.

**Differenze rispetto a ByteTrack (Giorno 1):**
- ByteTrack è ottimizzato per velocità e precisione nell'assegnazione degli ID — ideale per conteggio e detection real-time
- Norfair dà accesso diretto a `obj.estimate` (posizione stimata dal filtro di Kalman) e all'intera storia della traccia; supporta funzioni di distanza personalizzabili oltre a euclidea e IoU

**Perché analizzare le traiettorie?**
La sola detection dice *quanti* oggetti ci sono e *dove* si trovano adesso. Le traiettorie aggiungono la dimensione temporale e aprono scenari di analisi completamente diversi:
- Identificare pattern ricorrenti (quali corsie usano prevalentemente i camion?)
- Rilevare comportamenti anomali (veicolo fermo in zona di divieto, percorso inverso)
- Analizzare i flussi di traffico su finestre temporali più lunghe
- Costruire heatmap di densità del movimento per decisioni urbanistiche

In [ ]:
import norfair
from norfair import Detection, Tracker

### Conversione detection YOLO per Norfair

La cella seguente definisce una funzione che trasforma le **detection generate da YOLO** nel formato richiesto da **Norfair**.  

- Per ogni bounding box, viene calcolato il **centro** che servirà come punto tracciato.  
- Solo le detection con **confidenza ≥ soglia** vengono considerate.  
- Restituisce una lista di oggetti `Detection` con punti, punteggi e bounding box associati, pronta per il tracking con Norfair.

In [ ]:
def yolo_a_norfair(boxes_yolo, soglia_conf: float = 0.3) -> list:
    """
    Converte le detection YOLO nel formato atteso da Norfair.
    Usa il centro del bounding box come punto tracciato.
    """
    detections = []
    for box in boxes_yolo:
        conf = float(box.conf[0])
        if conf < soglia_conf:
            continue
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        detections.append(
            Detection(
                points=np.array([[cx, cy]]),
                scores=np.array([conf]),
                data={'bbox': [x1, y1, x2, y2]}
            )
        )
    return detections

**Tracking con Norfair e raccolta traiettorie.** Ad ogni frame, le detection YOLO vengono passate a Norfair che le associa ai track esistenti tramite distanza euclidea tra centroidi. La posizione centrale `(cx, cy)` stimata dal filtro di Kalman viene accumulata in `traiettorie[tid]` — una lista crescente di punti che rappresenta il percorso completo del veicolo nell'inquadratura. Al termine del loop, `traiettorie` è un dizionario `{track_id: [(x0,y0), (x1,y1), ...]}` che usiamo come base per tutte le analisi successive.

In [ ]:
N_FRAME_TRAJ = 200
OUTPUT_TRAJ     = str(VIDEOS_DIR / 'day2_traiettorie_norfair.mp4')

tracker_norfair = Tracker(
    distance_function='euclidean',
    distance_threshold=50,
    hit_counter_max=15,
    initialization_delay=2,
)

traiettorie = defaultdict(list)

cap = cv2.VideoCapture(VIDEO_AIC21)
fps_nf = cap.get(cv2.CAP_PROP_FPS)
W_nf   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H_nf   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

writer = cv2.VideoWriter(OUTPUT_TRAJ, cv2.VideoWriter_fourcc(*'H264'),
                         fps_nf, (W_nf, H_nf))

np.random.seed(42)
PALETTE_NF = np.random.randint(0, 255, size=(200, 3), dtype=np.uint8)

for i in tqdm(range(N_FRAME_TRAJ), desc='Trajectory extraction'):
    ret, frame = cap.read()
    if not ret:
        break

    ris = model_yolo(frame, verbose=False, classes=[2, 5, 7])[0]
    dets_nf = yolo_a_norfair(ris.boxes, soglia_conf=0.3)
    tracked = tracker_norfair.update(detections=dets_nf)

    out_frame = frame.copy()
    for obj in tracked:
        tid = obj.id
        cx, cy = obj.estimate[0]
        traiettorie[tid].append((int(cx), int(cy)))

        colore = tuple(int(c) for c in PALETTE_NF[tid % len(PALETTE_NF)])

        if obj.last_detection is not None and obj.last_detection.data:
            x1,y1,x2,y2 = map(int, obj.last_detection.data['bbox'])
            cv2.rectangle(out_frame, (x1,y1), (x2,y2), colore, 2)
            cv2.putText(out_frame, f'ID:{tid}', (x1, y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, colore, 2)

        punti_traj = traiettorie[tid][-30:]
        for j in range(1, len(punti_traj)):
            alpha = j / len(punti_traj)
            c_fade = tuple(int(c * alpha) for c in colore)
            cv2.line(out_frame, punti_traj[j-1], punti_traj[j], c_fade, 2)

        cv2.circle(out_frame, (int(cx), int(cy)), 5, colore, -1)

    cv2.putText(out_frame, f'Track attivi: {len(tracked)}',
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
    writer.write(out_frame)

cap.release()
writer.release()
print(f'Traiettorie raccolte: {len(traiettorie)}')
print(f'Durata media: {np.mean([len(v) for v in traiettorie.values()]):.1f} frame')
Video(OUTPUT_TRAJ, embed=True)

### 4.2 Analisi delle traiettorie

Raccogliere le traiettorie è solo il primo passo. L'analisi che segue risponde a domande concrete:

- **Durata**: per quanti frame mediamente un veicolo rimane nell'inquadratura? Dipende dalla velocità reale, dalla lunghezza del percorso nella scena e dall'angolo della telecamera.
- **Velocità media**: distanza percorsa in pixel per frame. Senza calibrazione telecamera è una misura relativa, ma permette già di distinguere veicoli lenti da veloci e di rilevare anomalie.
- **Heatmap di densità**: dove si concentra il traffico nel tempo? Sovrapposta al frame di riferimento, rivela i percorsi più battuti e le zone di accumulo.

In [ ]:
frame_sfondo = estrai_frame(VIDEO_AIC21, n=1)[0]
sfondo = (frame_sfondo.astype(np.float32) * 0.4).astype(np.uint8)

fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(cv2.cvtColor(sfondo, cv2.COLOR_BGR2RGB))

cmap_traj = cm.get_cmap('tab20')
for idx, (tid, punti) in enumerate(traiettorie.items()):
    if len(punti) < 5:
        continue
    xs = [p[0] for p in punti]
    ys = [p[1] for p in punti]
    ax.plot(xs, ys, linewidth=1.5, alpha=0.8, color=cmap_traj(idx % 20))
    ax.plot(xs[-1], ys[-1], 'o', markersize=4, color=cmap_traj(idx % 20))

ax.set_title(f'Traiettorie — {len(traiettorie)} veicoli tracciati', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()


### Heatmap di densità del traffico

In [ ]:
tutti_i_punti = [p for punti in traiettorie.values() for p in punti]
heatmap = crea_heatmap(tutti_i_punti, shape=(H_nf, W_nf), sigma=25)
frame_hm = sovrapponi_heatmap(estrai_frame(VIDEO_AIC21, n=1)[0], heatmap, alpha=0.6)
mostra_frame(frame_hm, 'Heatmap densità traffico — AIC21')


**Analisi statistica delle traiettorie.** La distribuzione delle durate mostra quanti veicoli sono stati tracciati per pochi frame (entrati e usciti rapidamente) rispetto a quanti hanno percorso l'intera scena. La velocità media in pixel/frame è proporzionale alla velocità reale se la telecamera è fissa — per convertirla in km/h servirebbe la calibrazione spaziale (pixel/metro), che in sistemi reali si ottiene con un oggetto di dimensioni note nel campo visivo o con le coordinate GPS della telecamera.

In [ ]:
# Distribuzione lunghezze e velocità — grafici interattivi
import plotly.graph_objects as go
from plotly.subplots import make_subplots

lunghezze = [len(v) for v in traiettorie.values()]
velocita  = []
for punti in traiettorie.values():
    if len(punti) < 2:
        continue
    dists = [np.linalg.norm(np.array(punti[i]) - np.array(punti[i-1]))
             for i in range(1, len(punti))]
    velocita.append(np.mean(dists))

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Distribuzione durata traiettorie',
                                    'Distribuzione velocità media'))

fig.add_trace(go.Histogram(
    x=lunghezze, nbinsx=20,
    marker_color='steelblue', marker_line_color='white',
    marker_line_width=1, opacity=0.85, name='Durata (frame)'
), row=1, col=1)

fig.add_trace(go.Histogram(
    x=velocita, nbinsx=20,
    marker_color='darkorange', marker_line_color='white',
    marker_line_width=1, opacity=0.85, name='Velocità (px/frame)'
), row=1, col=2)

fig.update_layout(height=380, showlegend=False, margin=dict(l=10, r=10, t=60, b=40))
fig.update_xaxes(title_text='Durata (frame)', row=1, col=1)
fig.update_xaxes(title_text='Velocità media (pixel/frame)', row=1, col=2)
fig.update_yaxes(title_text='N° oggetti', row=1, col=1)
fig.update_yaxes(title_text='N° oggetti', row=1, col=2)
fig.show()

print(f'Durata media: {np.mean(lunghezze):.1f} frame')
print(f'Velocità media: {np.mean(velocita):.1f} pixel/frame')


---
## 5. Riepilogo e discussione

### Cosa abbiamo fatto oggi:
1. **Vehicle counting** con `supervision` LineZone e PolygonZone su AIC21
2. **Robustezza meteorologica**: confronto detection in condizioni normali / pioggia / alba
3. **ShanghaiTech**: annotazioni `.mat` → density map GT → confronto con YOLO
4. **Trajectory extraction** con norfair → heatmap, analisi velocità

### Punti chiave:
- **LineZone / PolygonZone**: strumenti potenti per il counting senza retraining
- **YOLO su folla densa**: errori tipici > 80% → necessari modelli specializzati
- **Density map**: rappresentazione alternativa alle bounding box per la folla
- **Traiettorie**: permettono analisi comportamentale non possibili con la sola detectio